In [10]:
from pinnsform.util import *
from pinnsform.model import PINN

In [11]:
problem_domain = ([-1, 1], [0, 2*np.pi], [0, 60])

In [12]:
train_points = (16, 24, 61)

device = 'cuda'


np_mesh = generate_mesh(train_points, problem_domain)

sequence_mesh = make_temporal_sequence(np_mesh, num_step=5, step=0.2)

np_mesh_sequence = listify_sequence(sequence_mesh)

mesh = torchify(np_mesh_sequence, device, False)
boundary = torchified_borders(np_mesh_sequence, problem_domain, device, False)

#mesh, boundaries = generate_mesh_object(train_points, domain=problem_domain, device=device, full_requires_grad=True, border_requires_grad=False)

#b_left = boundaries[0][0]
#b_right = boundaries[0][1]
#initial = boundaries[1][0]

In [13]:
bottom  = boundary[0][0]
top     = boundary[0][1]
left    = boundary[1][0]
right   = boundary[1][1]
initial = boundary[2][0]

In [14]:
top.full

tensor([[ 1.0000,  0.0000,  0.0000],
        [ 1.0000,  0.0000,  0.2000],
        [ 1.0000,  0.0000,  0.4000],
        ...,
        [ 1.0000,  6.2832, 60.4000],
        [ 1.0000,  6.2832, 60.6000],
        [ 1.0000,  6.2832, 60.8000]], device='cuda:0')

In [15]:
def loss_fn(model, mesh, b_left, b_right, initial, initial_values):
    # pde
    pde_residue = df(model, mesh, wrt=1, order=2) - BETA*df(model, mesh, wrt=0, order=2)
    pde_loss = pde_residue.pow(2).mean()

    # boundary
    bleft_residue = f(model, b_left)
    bright_residue = f(model, b_right)
    boundary_loss = bleft_residue.pow(2).mean() + bright_residue.pow(2).mean()

    # initial value
    initialV_residue = f(model, initial) - initial_values
    
    # initial derivative
    initialD_residue = df(model, initial, wrt=1)

    initialV_loss = initialV_residue.pow(2).mean()
    initialD_loss = initialD_residue.pow(2).mean()

    return pde_loss, boundary_loss, initialV_loss, initialD_loss

In [16]:
base_model = PINN(in_dim=3, hidden_dim=512, out_dim=4, num_layer=4).to(device)

In [ ]:
# wrt
#   0   x
#   1   y
#   2   t

# of
#   0   u
#   1   v
#   2   p
#   3   T

u = f(model, of=0)
v = f(model, of=1)
p = f(model, of=2)
T = f(model, of=3)


Pr = 1.0
Ra = 1.0

np.sqrt(Pr/Ra)

df(model, of=0, wrt=2) + u*df(model, of=0, wrt=0) + v*df(model, of=0, wrt=1) - np.sqrt(Pr/Ra)*(df(model, of=0, wrt=0, order=2) + df(model, of=0, wrt=1, order=2)) - T + df(model, of=2, wrt=0)


# 
df(model, of=0, wrt=0) + df(model, of=1, wrt=1)